<a href="https://colab.research.google.com/github/busycaesar/Embeddings_And_Cosine_Similarity/blob/Master/AgentCon%20-%20Toronto/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required dependencies

In [2]:
%pip install langchain_community unstructured

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is 

Import environment variables

In [ ]:
from google.colab import userdata
from google.colab import auth

vector_store_address = userdata.get("AZURE_SEARCH_ENDPOINT")
vector_store_password = userdata.get("AZURE_SEARCH_ADMIN_KEY")

auth.authenticate_user()

Fetch the data

In [3]:
from langchain_community.document_loaders import UnstructuredURLLoader

urls = [
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-900',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-102',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-300',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/dp-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-731',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-730',
]

loader = UnstructuredURLLoader(urls)

documents = loader.load()

ImportError: unstructured package not found, please install it with `pip install unstructured`

Split the data into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

Store the data into vector database

In [ ]:
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

vector_store = AzureSearch(
    azure_search_endpoint=vector_store_address,
    azure_search_key=vector_store_password,
    index_name='consine-similarity-demo',
    embedding_function=embeddings.embed_query
)

vector_store.add_documents(chunks)

User's query

In [ ]:
user_query = "What are the Core components for building AI agents?"

Fetch the relevant chunk of data

In [ ]:
retrieved_docs = vector_store.similarity_search(query=user_query, k=3)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

Create prompt template

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"],
    template=
    """
        You are a helpful cloud instructor that provides cloud project ideas about Microsoft Azure Certifications based on the certification guide.

        Study Guide: {relevant_chunk_of_data}

        User's Question: {prompt}
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [ ]:
from langchain_openai import OpenAI
from IPython.display import clear_output

llm = OpenAI()

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

clear_output(wait=True)

print(response.content)